# PyTorch Delaney Regression Baseline

This notebook mirrors the Delaney regression baseline in PyTorch with a small multilayer perceptron over tabular molecular descriptors.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

sns.set_theme(style="whitegrid")
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)

In [ ]:
def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == 'notebooks' else cwd


PROJECT_ROOT = resolve_project_root()
DATA_DIR = PROJECT_ROOT / 'data'
delaney = pd.read_csv(DATA_DIR / 'delaney-processed.csv')
feature_columns = [
    'ESOL predicted log solubility in mols per litre',
    'Minimum Degree',
    'Molecular Weight',
    'Number of H-Bond Donors',
    'Number of Rings',
    'Number of Rotatable Bonds',
    'Polar Surface Area',
]
target_column = 'measured log solubility in mols per litre'
display(delaney.head())

In [ ]:
X = delaney[feature_columns]
y = delaney[target_column]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_SEED
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RANDOM_SEED
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_valid_tensor = torch.tensor(X_valid_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32).unsqueeze(1)
y_valid_tensor = torch.tensor(y_valid.to_numpy(), dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.float32).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=64, shuffle=True)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = nn.Sequential(
    nn.Linear(X_train_tensor.shape[1], 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1),
).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
history = []

for epoch in range(120):
    model.train()
    train_loss_total = 0.0
    for features, labels in train_loader:
        features = features.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        prediction = model(features)
        loss = criterion(prediction, labels)
        loss.backward()
        optimizer.step()
        train_loss_total += float(loss.item()) * len(features)

    model.eval()
    with torch.no_grad():
        valid_prediction = model(X_valid_tensor.to(device)).cpu().numpy().ravel()
    history.append({
        'epoch': epoch + 1,
        'train_mse': train_loss_total / len(X_train_tensor),
        'valid_rmse': float(np.sqrt(mean_squared_error(y_valid, valid_prediction))),
        'valid_mae': float(mean_absolute_error(y_valid, valid_prediction)),
        'valid_r2': float(r2_score(y_valid, valid_prediction)),
    })

history_df = pd.DataFrame(history)
display(history_df.tail().round(4))

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history_df['epoch'], history_df['train_mse'], label='train_mse')
plt.plot(history_df['epoch'], history_df['valid_rmse'], label='valid_rmse')
plt.title('PyTorch Delaney Training History')
plt.xlabel('Epoch')
plt.ylabel('Loss / RMSE')
plt.legend()
plt.show()

In [ ]:
model.eval()
with torch.no_grad():
    test_prediction = model(X_test_tensor.to(device)).cpu().numpy().ravel()

test_metrics = pd.DataFrame([
    {
        'rmse': float(np.sqrt(mean_squared_error(y_test, test_prediction))),
        'mae': float(mean_absolute_error(y_test, test_prediction)),
        'r2': float(r2_score(y_test, test_prediction)),
    }
])
display(test_metrics.round(4))

plot_df = pd.DataFrame({'true': y_test, 'predicted': test_prediction})
plt.figure(figsize=(6, 6))
sns.scatterplot(data=plot_df, x='true', y='predicted', s=40)
line_min = min(plot_df['true'].min(), plot_df['predicted'].min())
line_max = max(plot_df['true'].max(), plot_df['predicted'].max())
plt.plot([line_min, line_max], [line_min, line_max], color='black', linestyle='--')
plt.title('PyTorch Delaney Test Predictions')
plt.show()